# Import

In [ ]:
from foundry.transforms import Dataset
import pandas as pd
import random
import uuid
from faker import Faker
from datetime import datetime, timedelta

# Helpers

In [ ]:
fake = Faker()
random.seed(42)

In [ ]:
def parse_date(x):
    try:
        return datetime.strptime(str(x)[:10], "%Y-%m-%d")
    except:
        return None

def random_date(start, end):
    # start/end are datetime
    return start + timedelta(seconds=random.randint(0, int((end - start).total_seconds())))

# Load Disaster Event (your real dataset)

In [ ]:
disasters_df = Dataset.get("disaster_event").read_table(format="pandas")

# Basic cleanup + safe columns
disasters_df["Start_Date_dt"] = disasters_df["Start_Date"].apply(parse_date) if "Start_Date" in disasters_df.columns else None
disasters_df["End_Date_dt"] = disasters_df["End_Date"].apply(parse_date) if "End_Date" in disasters_df.columns else None

country_col = "_newCountry" if "_newCountry" in disasters_df.columns else ("Country" if "Country" in disasters_df.columns else None)

disaster_ids = disasters_df["disaster_id"].dropna().astype(str).unique().tolist()

# Simple mission generation config

In [ ]:
num_missions = 10000

org_by_type = {
    "Flood": ["WFP", "UNICEF", "Red Cross", "USAID"],
    "Storm": ["Red Cross", "WFP", "UNICEF", "USAID"],
    "Earthquake": ["Doctors Without Borders", "Red Cross", "USAID", "UNICEF"],
    "Drought": ["WFP", "UNICEF", "USAID"],
    "Epidemic": ["Doctors Without Borders", "UNICEF", "USAID", "Red Cross"],
}

default_orgs = ["Red Cross", "UNICEF", "WFP", "Doctors Without Borders", "USAID"]

supplies_by_type = {
    "Flood": ["Water Purification Tablets", "Hygiene Kits", "Temporary Shelters", "Food Rations"],
    "Storm": ["Medical Kits", "Tarps", "Water Purification Tablets", "Emergency Power Units"],
    "Earthquake": ["Trauma Medical Kits", "Search & Rescue Gear", "Temporary Shelters", "Water"],
    "Drought": ["Food Rations", "Water Containers", "Nutrition Supplements", "Livelihood Support Kits"],
    "Epidemic": ["PPE", "Testing Supplies", "Medicines", "Cold Chain Equipment"],
}

default_supplies = ["Medical Kits", "Water Purification Tablets", "Food Rations"]

mission_statuses = ["Planned", "En Route", "On Site", "Completed"]
priority_levels = ["Low", "Medium", "High", "Critical"]

# Build Relief Missions dataset

In [ ]:
missions_data = {
    "mission_id": [],
    "disaster_id": [],
    "mission_name": [],
    "organization": [],
    "priority_level": [],
    "mission_status": [],
    "country": [],
    "disaster_type": [],
    "personnel_count": [],
    "supplies_shipped": [],
    "last_updated_by": [],
    "mission_notes": [],
}

for _ in range(num_missions):
    # Pick a real disaster row so the mission matches it
    r = disasters_df.sample(1).iloc[0]

    disaster_id = str(r["disaster_id"])
    display_name = str(r["Display_Name"]) if "Display_Name" in disasters_df.columns and pd.notna(r.get("Display_Name")) else ""
    dtype = str(r["Disaster_Type"]) if "Disaster_Type" in disasters_df.columns and pd.notna(r.get("Disaster_Type")) else None

    # org + supplies based on disaster type (fallback to defaults)
    org = random.choice(org_by_type.get(dtype, default_orgs))
    supplies = supplies_by_type.get(dtype, default_supplies)
    supplies_shipped = ", ".join(random.sample(supplies, k=min(2, len(supplies))))

    # Dates: try to anchor around disaster start/end
    start_dt = r.get("Start_Date_dt")
    end_dt = r.get("End_Date_dt")

    if isinstance(start_dt, datetime):
        mission_start = start_dt + timedelta(days=random.randint(0, 14))
    else:
        mission_start = random_date(datetime.now() - timedelta(days=3650), datetime.now())

    mission_end = mission_start + timedelta(days=random.randint(3, 45))

    if isinstance(end_dt, datetime):
        # keep within a week after disaster end
        max_end = end_dt + timedelta(days=7)
        if mission_end > max_end:
            mission_end = max_end

    missions_data["mission_id"].append(str(uuid.uuid4()))
    missions_data["disaster_id"].append(disaster_id)
    missions_data["mission_name"].append(f"{org} Response - {display_name}" if display_name else f"{org} Response")
    missions_data["organization"].append(org)
    missions_data["priority_level"].append(random.choice(priority_levels))
    missions_data["mission_status"].append(random.choice(mission_statuses))
    missions_data["country"].append(r.get(country_col) if country_col else None)
    missions_data["disaster_type"].append(dtype)
    missions_data["personnel_count"].append(random.randint(5, 120))
    missions_data["supplies_shipped"].append(supplies_shipped)
    missions_data["last_updated_by"].append(fake.job().replace(",", ""))  # simple + readable
    missions_data["mission_notes"].append(fake.sentence(nb_words=10))

missions_df = pd.DataFrame(missions_data)

In [ ]:
missions_df.head(2)

In [ ]:
relief_missions = Dataset.get("relief_missions")
relief_missions.write_table(missions_df)